# 02 - Training
Fine-tuning backbone pretrained (timm) dengan PyTorch.

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/GANTI_USERNAME/document-image-classification.git"  # <-- GANTI
REPO_DIR = "/kaggle/working/document-image-classification"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
sys.path.append(REPO_DIR)

import pandas as pd, numpy as np, torch
from scripts.utils import set_seed, find_data_dir
set_seed(42)
DATA_DIR = find_data_dir()
CACHE_DIR = "/kaggle/working/cache"
IMG_SIZE = 224
train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
test_df = pd.read_csv(f"{DATA_DIR}/test.csv")
print(DATA_DIR, train_df.shape, test_df.shape)

In [ ]:
from scripts.preprocess import cache_images
cache_images(train_df, DATA_DIR, CACHE_DIR, IMG_SIZE)   # dilewati otomatis jika sudah ada
cache_images(test_df, DATA_DIR, CACHE_DIR, IMG_SIZE)

## Konfigurasi & DataLoader (split stratified 90/10)

In [ ]:
from sklearn.model_selection import train_test_split
from scripts.dataset import make_loader
CFG = dict(model="convnext_tiny", epochs=12, lr=2e-4, batch_size=64, label_smoothing=0.1, weight_decay=0.05)
tr_df, va_df = train_test_split(train_df, test_size=0.1, stratify=train_df["label"], random_state=42)
train_loader = make_loader(tr_df, CACHE_DIR, IMG_SIZE, CFG["batch_size"], train=True)
val_loader = make_loader(va_df, CACHE_DIR, IMG_SIZE, CFG["batch_size"], train=False)
print(len(tr_df), len(va_df))

## Model & training

In [ ]:
from scripts.model import build_model
from scripts.train import fit
device = "cuda"
model = build_model(CFG["model"], num_classes=train_df["label"].nunique())
history = fit(model, train_loader, val_loader, device, ckpt_dir="/kaggle/working/checkpoint",
              epochs=CFG["epochs"], lr=CFG["lr"], weight_decay=CFG["weight_decay"],
              label_smoothing=CFG["label_smoothing"], tag=CFG["model"])

## Kurva training & confusion matrix

In [ ]:
from scripts.utils import plot_history, plot_confusion_matrix
from scripts.train import evaluate
import torch.nn as nn
fig_dir = f"{REPO_DIR}/reports/figures"; os.makedirs(fig_dir, exist_ok=True)
plot_history(history, f"{fig_dir}/training_curve.png")
model.load_state_dict(torch.load(f"/kaggle/working/checkpoint/{CFG['model']}_best.pt"))
_, acc, preds, gts = evaluate(model, val_loader, nn.CrossEntropyLoss(), device)
names = train_df.drop_duplicates("label").sort_values("label")["class_name"].tolist()
print("Best val acc:", acc)
plot_confusion_matrix(gts, preds, names, f"{fig_dir}/confusion_matrix.png")
from sklearn.metrics import classification_report
print(classification_report(gts, preds, target_names=names, digits=3))